# Ejercicio 8: incorporación de 2025

Valida los archivos de 2025, ejecuta las consultas de los Ejercicios 4 a 8 sobre los tres años y muestra la evolución de los indicadores. Requiere haber ejecutado `scripts/download_data.py` y `scripts/construir_tablero.py`. La interpretación está en `docs/ejercicio8.md`.

In [1]:
import os
import time
from pathlib import Path

import duckdb
import pandas as pd

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

con = duckdb.connect()
con.execute(Path("sql/ejercicio4/00_vistas.sql").read_text())


def ejecutar(ruta: str, conexion=con) -> pd.DataFrame:
    return conexion.sql(Path(ruta).read_text()).df()

## 8.1 y 8.2 Archivos por año

Las consultas de validación del Ejercicio 5 leen todos los años con un glob, así que se usan sin cambios.

In [2]:
ejecutar("sql/ejercicio5/01_archivos_por_anio.sql")

,tipo,anio,archivos,primer_mes,ultimo_mes
0,green,2024,12,2024-01,2024-12
1,yellow,2024,12,2024-01,2024-12
2,green,2025,12,2025-01,2025-12
3,yellow,2025,12,2025-01,2025-12
4,green,2026,8,2026-01,2026-08
5,yellow,2026,8,2026-01,2026-08


In [3]:
ejecutar("sql/ejercicio5/02_manifiesto_vs_metadata.sql")

,tipo,anio,archivos_manifiesto,archivos_en_disco,filas_manifiesto,filas_metadata,diferencias
0,green,2024,12,12,660218.0,660218.0,0
1,yellow,2024,12,12,41169720.0,41169720.0,0
2,green,2025,12,12,591375.0,591375.0,0
3,yellow,2025,12,12,48722602.0,48722602.0,0
4,green,2026,8,8,337114.0,337114.0,0
5,yellow,2026,8,8,29703355.0,29703355.0,0


In [4]:
ejecutar("sql/ejercicio5/03_esquema_por_anio.sql")

,tipo,columna,detalle_por_anio
0,green,cbd_congestion_fee,2025: 12/12 archivos (DOUBLE) | 2026: 8/8 arch...
1,green,request_source,2026: 3/8 archivos (BYTE_ARRAY StringType())
2,yellow,cbd_congestion_fee,2025: 12/12 archivos (DOUBLE) | 2026: 8/8 arch...
3,yellow,request_source,2026: 3/8 archivos (BYTE_ARRAY StringType())


In [5]:
ejecutar("sql/ejercicio5/04_conteo_conjunto.sql")

,tipo,anio,archivos,registros
0,green,2024,12,660218
1,yellow,2024,12,41169720
2,green,2025,12,591375
3,yellow,2025,12,48722602
4,green,2026,8,337114
5,yellow,2026,8,29703355


## 8.3 Las consultas anteriores sobre los tres años

Se ejecutan sin modificar las consultas de los Ejercicios 4 a 8. Las del Ejercicio 6 usan la vista Parquet de `00_fuente_parquet.sql` sobre todos los años. Las del Ejercicio 3 fijan `/2026/` en la ruta y se mantienen como exploración de 2026 (Ej. 5, 5.7).

In [6]:
def probar(archivo: Path, conexion, **valores) -> dict:
    sql = archivo.read_text()
    for clave, valor in valores.items():
        sql = sql.replace("{" + clave + "}", valor)
    inicio = time.perf_counter()
    try:
        filas, error = len(conexion.sql(sql).fetchall()), ""
    except duckdb.Error as e:
        filas, error = None, str(e).splitlines()[0][:120]
    return {"consulta": f"{archivo.parent.name}/{archivo.stem}", "filas": filas,
            "segundos": round(time.perf_counter() - inicio, 1), "error": error}


con.execute(Path("sql/ejercicio6/00_fuente_parquet.sql").read_text()
            .replace("{vista}", "parquet_todos")
            .replace("{glob_yellow}", "data/raw/yellow/*/*.parquet")
            .replace("{glob_green}", "data/raw/green/*/*.parquet"))
con.execute("""CREATE OR REPLACE VIEW zonas_csv AS
               SELECT LocationID, Borough AS borough, Zone AS zona, service_zone
               FROM read_csv('data/raw/taxi_zone_lookup.csv', header = true)""")
tablero = duckdb.connect("data/processed/tablero.duckdb", read_only=True)

resultados = (
    [probar(a, con) for a in sorted(Path("sql/ejercicio4").glob("[01]*.sql")) if a.stem != "00_vistas"]
    + [probar(a, con) for a in sorted(Path("sql/ejercicio5").glob("*.sql"))]
    + [probar(a, con, fuente="parquet_todos", zonas="zonas_csv") for a in sorted(Path("sql/ejercicio6").glob("b*.sql"))]
    + [probar(a, tablero) for a in sorted(Path("sql").glob("ejercicio[78]/i*.sql"))]
)
prueba = pd.DataFrame(resultados)
prueba

,consulta,filas,segundos,error
0,ejercicio4/01_viajes_por_mes,64,2.4,
1,ejercicio4/02_hora_dia_semana,336,2.4,
2,ejercicio4/03_distribucion_viajes,8,16.5,
3,ejercicio4/04_velocidad_por_hora,48,5.5,
4,ejercicio4/05_pasajeros,18,2.6,
5,ejercicio4/06_zonas_servicio,20,2.8,
6,ejercicio4/07_top_zonas,20,3.1,
7,ejercicio4/08_aeropuertos,8,11.4,
8,ejercicio4/09_formas_de_pago,12,5.6,
9,ejercicio4/10_flex_fare_mensual,32,2.9,


In [7]:
prueba.assign(ok=prueba.error == "").groupby(prueba.consulta.str.split("/").str[0]).agg(
    consultas=("consulta", "count"), sin_error=("ok", "sum"), segundos=("segundos", "sum"))

,consultas,sin_error,segundos
consulta,,,
ejercicio4,17,17,99.9
ejercicio5,7,7,18.9
ejercicio6,6,6,15.0
ejercicio7,11,11,11.1
ejercicio8,2,2,1.3


## 8.4 y 8.5 Evolución de los indicadores

Indicadores del tablero sobre la base con los tres años.

In [8]:
ejecutar("sql/ejercicio7/i01_resumen_anual.sql", tablero)

,tipo,anio,meses,viajes_por_dia,ingresos_por_dia_miles_usd,total_mediano_usd,distancia_mediana_mi,duracion_mediana_min
0,yellow,2024,1 a 8,104863.0,2962.4,21.00,1.80,12.7
1,yellow,2025,1 a 8,124024.0,3383.0,21.57,1.90,13.1
2,yellow,2026,1 a 8,117693.0,3554.6,23.58,1.92,14.1
3,green,2024,1 a 8,1716.0,40.9,19.15,1.96,11.9
4,green,2025,1 a 8,1552.0,38.6,19.75,2.02,12.4
5,green,2026,1 a 8,1334.0,34.0,20.52,2.15,13.4


In [9]:
ejecutar("sql/ejercicio7/i02a_demanda_yellow.sql", tablero).pivot(index="anio", columns="mes", values="viajes_por_dia")

mes,1,2,3,4,5,6,7,8,9,10,11,12
anio,,,,,,,,,,,,
2024,92641.0,100216.0,111437.0,114281.0,117066.0,114716.0,96205.0,92667.0,116670.0,119201.0,117356.0,113818.0
2025,107377.0,122358.0,128298.0,127027.0,141204.0,137286.0,119460.0,109543.0,135095.0,136736.0,134161.0,132462.0
2026,114889.0,116106.0,122943.0,124124.0,127870.0,123198.0,109287.0,103359.0,NaN,NaN,NaN,NaN


In [10]:
ejecutar("sql/ejercicio7/i02b_demanda_green.sql", tablero).pivot(index="anio", columns="mes", values="viajes_por_dia")

mes,1,2,3,4,5,6,7,8,9,10,11,12
anio,,,,,,,,,,,,
2024,1726.0,1744.0,1751.0,1770.0,1862.0,1729.0,1570.0,1577.0,1717.0,1723.0,1644.0,1640.0
2025,1467.0,1567.0,1558.0,1626.0,1683.0,1573.0,1502.0,1442.0,1577.0,1538.0,1509.0,1500.0
2026,1254.0,1282.0,1376.0,1418.0,1396.0,1419.0,1274.0,1251.0,NaN,NaN,NaN,NaN


In [11]:
ejecutar("sql/ejercicio7/i04_velocidad_hora.sql", tablero).pivot(index="anio", columns="hora", values="velocidad_mediana_mph")

hora,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
anio,,,,,,,,,,,,,,,,,,,,,,,,
2024,11.80,12.46,12.73,13.09,13.71,13.48,12.59,10.32,8.39,7.68,7.36,7.05,7.08,7.30,7.25,7.30,7.55,7.66,8.09,8.82,9.50,9.91,10.18,10.60
2025,11.88,12.50,12.86,13.16,13.88,13.60,12.50,10.33,8.52,7.79,7.35,7.02,7.09,7.28,7.30,7.28,7.51,7.64,8.03,8.69,9.44,9.79,10.08,10.57
2026,11.29,12.08,12.55,12.93,13.53,13.35,12.14,9.96,8.19,7.38,6.91,6.56,6.61,6.78,6.79,6.76,6.99,7.17,7.58,8.19,8.92,9.23,9.49,9.96


In [12]:
ejecutar("sql/ejercicio7/i05_costo_viaje.sql", tablero).pivot(index="grupo", columns="canal", values="total_mediano_usd")

canal,Aplicación,Taxímetro
grupo,,
green 2024,25.18,18.90
green 2025,24.91,19.62
green 2026,26.47,19.90
yellow 2024,22.25,20.93
yellow 2025,22.80,21.42
yellow 2026,28.99,21.90


In [13]:
ejecutar("sql/ejercicio7/i06_formas_pago.sql", tablero).pivot(index="grupo", columns="forma", values="pct_viajes")

forma,Aplicación,Efectivo,Otros,Tarjeta
grupo,,,,
green 2024,4.01,27.78,0.45,67.75
green 2025,6.89,23.09,0.44,69.57
green 2026,14.71,19.41,0.34,65.54
yellow 2024,9.25,13.75,1.29,75.70
yellow 2025,22.48,9.66,1.60,66.26
yellow 2026,24.62,9.10,0.63,65.64


In [14]:
ejecutar("sql/ejercicio7/i07_propinas.sql", tablero).pivot(index="grupo", columns="tramo", values="pct_viajes")

tramo,1. sin propina,2. menos de 20%,3. 20% a 25%,4. 25% a 30%,5. 30% o más
grupo,,,,,
green 2024,8.81,20.45,30.40,22.85,17.49
green 2025,8.90,20.69,30.02,23.18,17.20
green 2026,8.66,19.06,30.93,23.68,17.66
yellow 2024,5.56,19.16,19.34,27.69,28.25
yellow 2025,6.58,18.36,15.89,26.44,32.73
yellow 2026,8.79,17.42,15.88,25.73,32.18


In [15]:
ejecutar("sql/ejercicio7/i08_aeropuertos.sql", tablero)

,grupo,pct_viajes,pct_ingresos
0,yellow 2024,10.14,28.04
1,yellow 2025,8.54,24.64
2,yellow 2026,8.15,21.05
3,green 2024,4.15,8.09
4,green 2025,3.64,7.14
5,green 2026,3.34,6.35


In [16]:
ejecutar("sql/ejercicio7/i09_zona_origen.sql", tablero).pivot(index="grupo", columns="origen", values="pct_viajes")

origen,Alto Manhattan,Brooklyn,Manhattan centro (Yellow Zone),Queens,Resto
grupo,,,,,
green 2024,53.28,13.40,7.16,24.87,1.29
green 2025,54.85,13.97,6.93,22.45,1.80
green 2026,52.51,15.81,6.95,22.01,2.72
yellow 2024,2.07,1.26,86.87,1.25,8.56
yellow 2025,2.96,3.54,83.17,2.60,7.73
yellow 2026,3.49,3.56,83.23,2.30,7.42


In [17]:
ejecutar("sql/ejercicio7/i10_calidad_mensual.sql", tablero).pivot(index="mes", columns="tipo", values="pct_excluidos")

tipo,green,yellow
mes,,
2024-01-01,5.39,3.13
2024-02-01,5.59,3.37
2024-03-01,5.55,3.58
2024-04-01,5.95,2.44
2024-05-01,5.40,2.55
2024-06-01,5.27,2.76
2024-07-01,6.10,3.07
2024-08-01,5.55,3.58
2024-09-01,5.41,3.66


In [18]:
ejecutar("sql/ejercicio8/i11_aplicacion_mensual.sql", tablero).pivot(index="mes", columns="tipo", values="pct_aplicacion")

tipo,green,yellow
mes,,
2024-01-01,6.11,4.08
2024-02-01,5.59,5.28
2024-03-01,3.76,10.99
2024-04-01,3.64,11.74
2024-05-01,3.18,10.97
2024-06-01,3.52,11.64
2024-07-01,3.16,9.10
2024-08-01,3.13,8.49
2024-09-01,3.18,12.76


In [19]:
ejecutar("sql/ejercicio8/i12_velocidad_mensual.sql", tablero)

,mes,velocidad_mediana_mph
0,2024-01-01,8.29
1,2024-02-01,8.02
2,2024-03-01,8.04
3,2024-04-01,7.88
4,2024-05-01,7.49
5,2024-06-01,7.72
6,2024-07-01,8.04
7,2024-08-01,8.12
8,2024-09-01,7.33
9,2024-10-01,7.25


## Flex Fare con tarifa negativa

Registros que el ajuste de `00_construir_tablas.sql` conserva sin montos, por mes.

In [20]:
ejecutar("sql/ejercicio8/01_flex_tarifa_negativa.sql")

,anio,mes,flex_fare,tarifa_negativa,pct_tarifa_negativa,pct_vendor2,distancia_negativa,distancia_resto,total_negativa,total_resto
0,2024,1,140162,2066,1.5,100.0,1.89,1.95,3.00,22.04
1,2024,2,185610,4656,2.5,100.0,1.90,1.98,3.00,21.26
2,2024,3,426190,14302,3.4,100.0,1.96,2.16,3.00,21.43
3,2024,4,408576,13985,3.4,100.0,2.06,2.49,3.00,21.86
4,2024,5,404666,11157,2.8,100.0,1.87,2.44,3.00,23.71
5,2024,6,410781,13345,3.2,100.0,1.86,2.37,3.00,23.13
6,2024,7,278989,8646,3.1,100.0,1.86,2.31,3.00,21.67
7,2024,8,258667,5667,2.2,100.0,1.77,2.27,2.88,21.78
8,2024,9,483731,17661,3.7,100.0,1.78,2.30,3.00,22.24
9,2024,10,393895,14511,3.7,100.0,1.77,2.28,3.00,21.98
